# Skin Disease Classifier — Complete Training Pipeline

This is a cleaned-up, working version of your project. It fixes the issues found in your
original notebook:

- **Kernel-restart crash** (`NameError: name 'model' is not defined`) — this notebook trains
  the model and uses it in one continuous flow, so there's nothing left undefined.
- **Two different preprocessing methods mixed together** (`rescale=1./255` vs
  `preprocess_input`) — this notebook uses `preprocess_input` everywhere, since that's what
  MobileNetV2 was actually trained on. Mixing them silently hurts accuracy.
- **No handling of class imbalance** — your `Unknown_Normal` class has 1651 images vs. 248
  for `Candidiasis`. Without correcting for this, the model just learns to favor the big
  classes. This notebook adds `class_weight`.
- **No callbacks** — training just ran a fixed number of epochs with no checkpointing, no
  early stopping, and no learning-rate decay, which is likely a big reason accuracy stalled
  around 36%.
- **Two-phase training done cleanly**: first train just the new classification head
  (base frozen), then unfreeze the top of MobileNetV2 and fine-tune with a low learning rate.

**Before running:** update `DATASET_ROOT` in the Config cell below to point at your dataset
folder (it assumes the same `dataset/SkinDisease/train` and `dataset/SkinDisease/test`
structure your original notebook used).

## 1. Imports

In [2]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing import image as keras_image

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


## 2. Config — edit these paths/settings for your machine

In [3]:
DATASET_ROOT = "dataset/SkinDisease"
TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
TEST_DIR = os.path.join(DATASET_ROOT, "test")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Set to False to drop "Unknown_Normal" (healthy skin) and classify diseases only,
# like your second attempt did. Keeping it True lets the model also say "this looks normal".
INCLUDE_UNKNOWN_NORMAL = True

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

assert os.path.exists(TRAIN_DIR), f"Train folder not found: {TRAIN_DIR}"
assert os.path.exists(TEST_DIR), f"Test folder not found: {TEST_DIR}"
print("Dataset folders found.")

Dataset folders found.


## 3. Classes

In [4]:
all_classes = sorted([
    d for d in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, d))
])

classes_to_use = all_classes if INCLUDE_UNKNOWN_NORMAL else [
    c for c in all_classes if c != "Unknown_Normal"
]

print(f"Using {len(classes_to_use)} classes:")
for i, c in enumerate(classes_to_use, start=1):
    print(f"{i}. {c}")

Using 22 classes:
1. Acne
2. Actinic_Keratosis
3. Benign_tumors
4. Bullous
5. Candidiasis
6. DrugEruption
7. Eczema
8. Infestations_Bites
9. Lichen
10. Lupus
11. Moles
12. Psoriasis
13. Rosacea
14. Seborrh_Keratoses
15. SkinCancer
16. Sun_Sunlight_Damage
17. Tinea
18. Unknown_Normal
19. Vascular_Tumors
20. Vasculitis
21. Vitiligo
22. Warts


## 4. Data generators

Uses `preprocess_input` from `mobilenet_v2` (matches what the pretrained network expects)
instead of a manual `rescale=1./255` — this alone should meaningfully help accuracy versus
your original run.

In [5]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    classes=classes_to_use,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=42
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    classes=classes_to_use,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

num_classes = len(train_generator.class_indices)
print("\nNumber of classes:", num_classes)
print("Class indices:", train_generator.class_indices)

# Save the mapping now — you'll need this exact mapping later to interpret predictions,
# and it's easy to lose track of once you reload a saved model in a new session.
with open(os.path.join(MODEL_DIR, "class_indices.json"), "w") as f:
    json.dump(train_generator.class_indices, f, indent=2)
print("\nSaved class_indices.json")

Found 13898 images belonging to 22 classes.
Found 1546 images belonging to 22 classes.

Number of classes: 22
Class indices: {'Acne': 0, 'Actinic_Keratosis': 1, 'Benign_tumors': 2, 'Bullous': 3, 'Candidiasis': 4, 'DrugEruption': 5, 'Eczema': 6, 'Infestations_Bites': 7, 'Lichen': 8, 'Lupus': 9, 'Moles': 10, 'Psoriasis': 11, 'Rosacea': 12, 'Seborrh_Keratoses': 13, 'SkinCancer': 14, 'Sun_Sunlight_Damage': 15, 'Tinea': 16, 'Unknown_Normal': 17, 'Vascular_Tumors': 18, 'Vasculitis': 19, 'Vitiligo': 20, 'Warts': 21}

Saved class_indices.json


## 5. Class weights (handles the imbalance)

Your dataset ranges from 248 images (`Candidiasis`) to 1651 images (`Unknown_Normal`) per
class. `class_weight='balanced'` makes the loss penalize mistakes on rare classes more,
so the model doesn't just learn to predict the biggest classes.

In [6]:
y_train = train_generator.classes
class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weight_values))

idx_to_class = {v: k for k, v in train_generator.class_indices.items()}
print("Class weights:")
for idx, w in class_weights.items():
    print(f"  {idx_to_class[idx]:25s}: {w:.2f}")

Class weights:
  Acne                     : 1.07
  Actinic_Keratosis        : 0.84
  Benign_tumors            : 0.58
  Bullous                  : 1.25
  Candidiasis              : 2.55
  DrugEruption             : 1.15
  Eczema                   : 0.63
  Infestations_Bites       : 1.21
  Lichen                   : 1.14
  Lupus                    : 2.03
  Moles                    : 1.75
  Psoriasis                : 0.77
  Rosacea                  : 2.49
  Seborrh_Keratoses        : 1.39
  SkinCancer               : 0.91
  Sun_Sunlight_Damage      : 2.02
  Tinea                    : 0.68
  Unknown_Normal           : 0.38
  Vascular_Tumors          : 1.16
  Vasculitis               : 1.37
  Vitiligo                 : 0.88
  Warts                    : 1.09


## 6. Build the model (MobileNetV2 base + custom head)

In [7]:
def build_model(num_classes):
    base_model = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base_model.trainable = False  # freeze for phase 1

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    outputs = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs=base_model.input, outputs=outputs)
    return model, base_model

model, base_model = build_model(num_classes)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1 (Conv2D)                │ (None, 112, 112, 32)      │             864 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bn_Conv1 (BatchNormalization) │ (None, 112, 112, 32)      │             128 │ Conv1[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1_relu (ReLU)             │ (None, 112, 112, 32)      │               0 │ bn_Conv1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise       │ (None, 112, 112, 32)      │             288 │ Conv1_relu[0][0]           │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_BN    │ (None, 112, 112, 32)      │             128 │ expanded_conv_depthwise[0… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_relu  │ (None, 112, 112, 32)      │               0 │ expanded_conv_depthwise_B… │
│ (ReLU)                        │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project         │ (None, 112, 112, 16)      │             512 │ expanded_conv_depthwise_r… │
│ (Conv2D)                      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project_BN      │ (None, 112, 112, 16)      │              64 │ expanded_conv_project[0][… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand (Conv2D)       │ (None, 112, 112, 96)      │           1,536 │ expanded_conv_project_BN[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_BN             │ (None, 112, 112, 96)      │             384 │ block_1_expand[0][0]       │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_relu (ReLU)    │ (None, 112, 112, 96)      │               0 │ block_1_expand_BN[0][0]    │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_pad (ZeroPadding2D)   │ (None, 113, 113, 96)      │               0 │ block_1_expand_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_depthwise             │ (None, 56, 56, 96)        │             864 │ block_1_pad[0][0]          │
│ (DepthwiseConv2D)             │                           │               

 Total params: 2,592,598 (9.89 MB)

 Trainable params: 334,102 (1.27 MB)

 Non-trainable params: 2,258,496 (8.62 MB)

## 7. Callbacks — checkpointing, early stopping, LR decay

In [8]:
checkpoint_path = os.path.join(MODEL_DIR, "best_model.keras")

callbacks = [
    ModelCheckpoint(checkpoint_path, monitor="val_accuracy", save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1),
]

## 8. Phase 1 — train the classification head (base frozen)

This trains fast since only the new top layers are updating. Aim for this to plateau before
moving to fine-tuning.

In [9]:
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

EPOCHS_HEAD = 10

history_head = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=EPOCHS_HEAD,
    class_weight=class_weights,
    callbacks=callbacks
)

Epoch 1/10
435/435 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.2465 - loss: 2.9199
Epoch 1: val_accuracy improved from None to 0.40168, saving model to models\best_model.keras

Epoch 1: finished saving model to models\best_model.keras
435/435 ━━━━━━━━━━━━━━━━━━━━ 1248s 3s/step - accuracy: 0.3035 - loss: 2.5937 - val_accuracy: 0.4017 - val_loss: 1.9486 - learning_rate: 0.0010
Epoch 2/10
435/435 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.3848 - loss: 2.1115
Epoch 2: val_accuracy improved from 0.40168 to 0.41203, saving model to models\best_model.keras

Epoch 2: finished saving model to models\best_model.keras
435/435 ━━━━━━━━━━━━━━━━━━━━ 1089s 3s/step - accuracy: 0.3884 - loss: 2.1043 - val_accuracy: 0.4120 - val_loss: 1.9237 - learning_rate: 0.0010
Epoch 3/10
435/435 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4158 - loss: 1.9907
Epoch 3: val_accuracy improved from 0.41203 to 0.42626, saving model to models\best_model.keras

Epoch 3: finished saving model to models\best_model.ker

## 9. Phase 2 — fine-tune the top of MobileNetV2

Unfreezes the last 40 layers of the base network and continues training with a much lower
learning rate, so we adapt the pretrained features to skin images without wrecking them.

In [10]:
base_model.trainable = True

FINE_TUNE_AT = len(base_model.layers) - 40
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

EPOCHS_FINE_TUNE = 15
initial_epoch = history_head.epoch[-1] + 1
total_epochs = initial_epoch + EPOCHS_FINE_TUNE

history_fine = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=total_epochs,
    initial_epoch=initial_epoch,
    class_weight=class_weights,
    callbacks=callbacks
)

Epoch 11/25
158/435 ━━━━━━━━━━━━━━━━━━━━ 13:02 3s/step - accuracy: 0.3442 - loss: 2.3994


KeyboardInterrupt



## 10. Save the final model

In [ ]:
final_path = os.path.join(MODEL_DIR, "skin_disease_final.keras")
model.save(final_path)
print("Final model saved to:", final_path)

## 11. Training curves

In [11]:
def combine_history(h1, h2):
    combined = {}
    for k in h1.history:
        combined[k] = h1.history[k] + h2.history.get(k, [])
    return combined

hist = combine_history(history_head, history_fine)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(hist["accuracy"], label="train")
plt.plot(hist["val_accuracy"], label="val")
plt.axvline(x=EPOCHS_HEAD, color="gray", linestyle="--", label="fine-tune start")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(hist["loss"], label="train")
plt.plot(hist["val_loss"], label="val")
plt.axvline(x=EPOCHS_HEAD, color="gray", linestyle="--", label="fine-tune start")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.show()

NameError: name 'history_fine' is not defined

## 12. Evaluation — per-class report + confusion matrix

In [ ]:
test_generator.reset()
predictions = model.predict(test_generator)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes
class_names = list(test_generator.class_indices.keys())

print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(14, 12))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xticks(range(len(class_names)), class_names, rotation=90)
plt.yticks(range(len(class_names)), class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

## 13. Inference — predict a single image

Once trained, this is the function your app (web page, script, whatever front-end you
build) would call.

In [ ]:
def predict_image(img_path, model, class_indices, top_k=3):
    idx_to_class = {v: k for k, v in class_indices.items()}

    img = keras_image.load_img(img_path, target_size=IMG_SIZE)
    img_array = keras_image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)

    preds = model.predict(img_array, verbose=0)[0]
    top_indices = preds.argsort()[-top_k:][::-1]

    print(f"Predictions for {img_path}:")
    for i in top_indices:
        print(f"  {idx_to_class[i]:25s}: {preds[i] * 100:.2f}%")

    best_idx = top_indices[0]
    return idx_to_class[best_idx], float(preds[best_idx])

# Example:
# predict_image("path/to/some_test_image.jpg", model, train_generator.class_indices)

## 14. Loading the saved model later (new session)

Run this in a fresh notebook/script once training is done — you don't need to retrain
every time you want to make a prediction.

In [ ]:
# from tensorflow.keras.models import load_model
# import json
#
# loaded_model = load_model("models/skin_disease_final.keras")
# with open("models/class_indices.json") as f:
#     class_indices = json.load(f)
#
# predict_image("path/to/image.jpg", loaded_model, class_indices)